In [28]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("USERNAME:", os.getenv("DB_USERNAME"))
print("PASSWORD:", os.getenv("DB_PASSWORD"))
print("HOST:",     os.getenv("DB_HOST"))
print("PORT:",     os.getenv("DB_PORT"))
print("DB:",       os.getenv("DB_NAME"))

USERNAME: root
PASSWORD: Root@1234
HOST: 127.0.0.1
PORT: 3306
DB: etl_db


In [2]:
#Building an ETL Data Pipeline
#Step 1: Extract data from csv file into a Pandas Dataframe
#Step 2: Transform the data(i.e clean the data) - Deal with missing and duplicate data
#Step 3: Create a database
#Step 4: Load the clean data into the database
import pandas as pd #For data Extract/transformation/manipulation/wrangling/analysis, etc
import psycopg2 #For Connecting Python to Postgresql database
from sqlalchemy import create_engine #To efficiently manage and reuse the database connections

In [7]:
#Step 1: Extract data from csv file into a Pandas Dataframe
#get the file path for the document/data
data = pd.read_csv("data/employees.csv")

In [8]:
#View the top five rows
print(data.head())

   employee_id         department     region   education gender  \
0         8724         Technology  region_26  Bachelor's      m   
1        74430                 HR   region_4  Bachelor's      f   
2        72255  Sales & Marketing  region_13  Bachelor's      m   
3        38562        Procurement   region_2  Bachelor's      f   
4        64486            Finance  region_29  Bachelor's      m   

  recruitment_channel  no_of_trainings  age  previous_year_rating  \
0            sourcing                1   24                   NaN   
1               other                1   31                   3.0   
2               other                1   31                   1.0   
3               other                3   31                   2.0   
4            sourcing                1   30                   4.0   

   length_of_service  awards_won  avg_training_score  
0                  1           0                  77  
1                  5           0                  51  
2                

In [9]:
#View the bottom five rows
print(data.tail())

       employee_id   department     region         education gender  \
23490        53478        Legal   region_2   Below Secondary      m   
23491        25600   Technology  region_25        Bachelor's      m   
23492        45409           HR  region_16        Bachelor's      f   
23493         1186  Procurement  region_31        Bachelor's      m   
23494         5973   Technology  region_17  Master's & above      m   

      recruitment_channel  no_of_trainings  age  previous_year_rating  \
23490            sourcing                1   24                   3.0   
23491            sourcing                1   31                   3.0   
23492            sourcing                1   26                   4.0   
23493            sourcing                3   27                   NaN   
23494               other                3   40                   5.0   

       length_of_service  awards_won  avg_training_score  
23490                  1           0                  61  
23491           

In [10]:
#Step 2: Transform the data(i.e clean the data) - Deal with missing and duplicate data
print(data.duplicated().sum())

5


In [11]:
#Remove duplicate records
data.drop_duplicates(keep="first", inplace=True)
print(data.duplicated().sum())

0


In [12]:
#Explore missing data
print(data.isnull().sum())

employee_id                0
department                 0
region                     0
education               1034
gender                     0
recruitment_channel        0
no_of_trainings            0
age                        0
previous_year_rating    1812
length_of_service          0
awards_won                 0
avg_training_score         0
dtype: int64


In [13]:
#Let's take a look at the missing data - EDUCATION COLUMN
print(data[data['education'].isnull()].head())

     employee_id         department     region education gender  \
49          3720  Sales & Marketing   region_2       NaN      m   
67         26977         Operations   region_2       NaN      m   
89          2347          Analytics  region_15       NaN      m   
127        42719          Analytics  region_21       NaN      m   
138         2721          Analytics  region_22       NaN      f   

    recruitment_channel  no_of_trainings  age  previous_year_rating  \
49             sourcing                1   36                   5.0   
67             sourcing                1   28                   3.0   
89                other                2   37                   5.0   
127            sourcing                1   27                   NaN   
138            sourcing                2   26                   5.0   

     length_of_service  awards_won  avg_training_score  
49                   4           0                  49  
67                   3           0                  59  

In [14]:
#Deal with missing data
data['education'] = data['education'].fillna('unknown') #For the education column
data['previous_year_rating'] = data['previous_year_rating'].fillna('0') #For the previous_year_rating column

In [15]:
data.isnull().sum()

employee_id             0
department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
awards_won              0
avg_training_score      0
dtype: int64

In [51]:
#Step 3: Create a database and establish a connection
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(override=True)

engine = create_engine(
    'mysql+mysqlconnector://',
    connect_args={
        "host":     os.getenv("DB_HOST"),
        "port":     int(os.getenv("DB_PORT")),  # must be int, not string
        "user":     os.getenv("DB_USERNAME"),
        "password": os.getenv("DB_PASSWORD"),   # @ in password is safe here
        "database": os.getenv("DB_NAME")
    }
)

try:
    with engine.connect() as conn:
        print("Connected successfully!")
except Exception as e:
    print(f"Failed: {e}")


Connected successfully!


In [52]:
#Load the database - employee_table
data.to_sql('employee_table', engine, if_exists='replace', index=False)
#Close the connection
engine.dispose()